In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

MODELS_DIR = PROJECT_ROOT / "models"

In [3]:
random_forest_model = joblib.load(
    MODELS_DIR / "random_forest_match_model.pkl"
)

In [4]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
candidate = {
    "roles": ["Python Developer"],
    "skills_primary": ["Python", "Django", "PostgreSQL"],
    "skills_secondary": ["Docker", "Redis"],
    "experience_years": 1,
    "domains": ["Software Development"],
    "career_intent": "Looking for backend Python developer roles"
}

In [6]:
job = {
    "title": "Junior Python Developer",
    "description": """
    We are looking for a Python developer with experience in Django,
    PostgreSQL and Docker. The candidate will work on backend APIs
    and web applications.
    """,
    "experience_years_hint": 1,
    "remote_hint": 0
}

In [7]:
candidate_skills = (
    candidate["skills_primary"]
    + candidate["skills_secondary"]
)

candidate_skill_count = len(
    set(skill.lower() for skill in candidate_skills)
)

print("Candidate skill count:", candidate_skill_count)

Candidate skill count: 5


In [8]:
candidate_experience_years = candidate["experience_years"]

expected_experience_years = job["experience_years_hint"]

experience_gap = (
    candidate_experience_years
    - expected_experience_years
)

experience_fit = (
    1 - abs(experience_gap) /
    (expected_experience_years + 1)
)

experience_fit = np.clip(
    experience_fit,
    0,
    1
)

print("Candidate experience:", candidate_experience_years)
print("Expected experience:", expected_experience_years)
print("Experience gap:", experience_gap)
print("Experience fit:", experience_fit)

Candidate experience: 1
Expected experience: 1
Experience gap: 0
Experience fit: 1.0


In [9]:
job_text = (
    job["title"] + " " +
    job["description"]
).lower()

candidate_skills_lower = [
    skill.lower()
    for skill in candidate_skills
]

matched_skills = [
    skill
    for skill in candidate_skills_lower
    if skill in job_text
]

matched_skill_count = len(
    set(matched_skills)
)

job_skill_count = len(
    set(matched_skills)
)

print("Matched skills:", matched_skills)
print("Matched skill count:", matched_skill_count)
print("Job skill count:", job_skill_count)

Matched skills: ['python', 'django', 'postgresql', 'docker']
Matched skill count: 4
Job skill count: 4


In [10]:
if job_skill_count > 0:
    job_skill_coverage = (
        matched_skill_count /
        job_skill_count
    )
else:
    job_skill_coverage = 0

if candidate_skill_count > 0:
    candidate_skill_coverage = (
        matched_skill_count /
        candidate_skill_count
    )
else:
    candidate_skill_coverage = 0

print("Job skill coverage:", job_skill_coverage)
print("Candidate skill coverage:", candidate_skill_coverage)

Job skill coverage: 1.0
Candidate skill coverage: 0.8


In [11]:
TRAINING_DIR = PROJECT_ROOT / "data" / "training"

training_data = pd.read_csv(
    TRAINING_DIR / "candidate_job_features.csv"
)

print(training_data.shape)

(5000, 16)


In [12]:
cleaned_data = pd.read_pickle(
    TRAINING_DIR / "candidate_job_cleaned.pkl"
)

print(cleaned_data.shape)

(5000, 21)


In [13]:
skill_vocabulary = set()

for skills in cleaned_data["candidate_skills_primary"]:
    if isinstance(skills, list):
        for skill in skills:
            skill_vocabulary.add(skill.lower())

for skills in cleaned_data["candidate_skills_secondary"]:
    if isinstance(skills, list):
        for skill in skills:
            skill_vocabulary.add(skill.lower())

print("Skill vocabulary size:", len(skill_vocabulary))

Skill vocabulary size: 782


In [14]:
job_skills = [
    skill
    for skill in skill_vocabulary
    if skill in job_text
]

job_skill_count = len(job_skills)

matched_skills = [
    skill
    for skill in job_skills
    if skill in candidate_skills_lower
]

matched_skill_count = len(
    matched_skills
)

print("Job skills:", job_skills)
print("Matched skills:", matched_skills)

Job skills: ['postgresql', 'django', 'docker', 'go', 'r', 'python', 'sql']
Matched skills: ['postgresql', 'django', 'docker', 'python']


In [15]:
if job_skill_count > 0:
    job_skill_coverage = (
        matched_skill_count /
        job_skill_count
    )
else:
    job_skill_coverage = 0

if candidate_skill_count > 0:
    candidate_skill_coverage = (
        matched_skill_count /
        candidate_skill_count
    )
else:
    candidate_skill_coverage = 0

print("Job skill coverage:", job_skill_coverage)
print("Candidate skill coverage:", candidate_skill_coverage)

Job skill coverage: 0.5714285714285714
Candidate skill coverage: 0.8


In [16]:
candidate_role_text = " ".join(
    candidate["roles"]
).lower()

job_title_words = set(
    job["title"].lower().split()
)

candidate_role_words = set(
    candidate_role_text.split()
)

if job_title_words:
    role_match_score = (
        len(job_title_words & candidate_role_words)
        / len(job_title_words)
    )
else:
    role_match_score = 0

print("Role match score:", role_match_score)

Role match score: 0.6666666666666666


In [17]:
candidate_text = " ".join(
    candidate["roles"]
    + candidate["skills_primary"]
    + candidate["skills_secondary"]
    + candidate["domains"]
    + [candidate["career_intent"]]
)

candidate_embedding = embedding_model.encode(
    [candidate_text],
    convert_to_numpy=True
)[0]

job_embedding = embedding_model.encode(
    [job["title"] + " " + job["description"]],
    convert_to_numpy=True
)[0]

semantic_similarity = (
    np.dot(candidate_embedding, job_embedding)
    / (
        np.linalg.norm(candidate_embedding)
        * np.linalg.norm(job_embedding)
    )
)

print("Semantic similarity:", semantic_similarity)

Semantic similarity: 0.8025207


In [18]:
prediction_features = pd.DataFrame([{
    "candidate_experience_years": candidate_experience_years,
    "candidate_skill_count": candidate_skill_count,
    "expected_experience_years": expected_experience_years,
    "experience_gap": experience_gap,
    "experience_fit": experience_fit,
    "job_skill_count": job_skill_count,
    "matched_skill_count": matched_skill_count,
    "job_skill_coverage": job_skill_coverage,
    "candidate_skill_coverage": candidate_skill_coverage,
    "role_match_score": role_match_score,
    "job_remote_hint": job["remote_hint"],
    "semantic_similarity": semantic_similarity
}])

prediction_features

,candidate_experience_years,candidate_skill_count,expected_experience_years,experience_gap,experience_fit,job_skill_count,matched_skill_count,job_skill_coverage,candidate_skill_coverage,role_match_score,job_remote_hint,semantic_similarity
0,1,5,1,0,1.0,7,4,0.571429,0.8,0.666667,0,0.802521


In [19]:
match_probability = random_forest_model.predict_proba(
    prediction_features
)[:, 1][0]

match_prediction = random_forest_model.predict(
    prediction_features
)[0]

print("Match probability:", match_probability)
print("Match prediction:", match_prediction)

Match probability: 0.665
Match prediction: 1


In [20]:
match_percentage = match_probability * 100

print(
    f"Candidate-Job Match: {match_percentage:.2f}%"
)

Candidate-Job Match: 66.50%


In [21]:
if match_probability >= 0.75:
    match_category = "Excellent Match"
elif match_probability >= 0.50:
    match_category = "Good Match"
elif match_probability >= 0.25:
    match_category = "Moderate Match"
else:
    match_category = "Poor Match"

print("Match category:", match_category)

Match category: Good Match


In [22]:
result = {
    "match_probability": round(match_percentage, 2),
    "match_category": match_category,
    "model_prediction": int(match_prediction)
}

print(result)

{'match_probability': np.float64(66.5), 'match_category': 'Good Match', 'model_prediction': 1}
